<CENTER><img src="images/logos.png" style="width:50%"></CENTER>

### Problem reading root files with uproot using https

Accessing the root files on the cern server with uproot has its problems. Root files have many 'baskets' within them. These baskets are compressed memory buffers which provide information about the data that ROOT holds. Large files can consist of many thousands of baskets. The http protocol wants to access all of these baskets in order to read the file. The CERN server has a hard limit on the number of basket reads, say about one hundred. Consquently the http protocol returns and error (error 429) and fails to read the root file when the cern server hits the limit for large file reads.

### Solution

There are two workarounds to access root files efficiently:

1. Use a protocol other than https, eg. root://
2. Provide local copies of the files and access those

We can use the module atlasopenmagic to find the path of files using the root:// protocol or we can download files locally for direct access via https   

Lets access the file http://opendata.cern.ch/eos/opendata/atlas/OutreachDatasets/2020-08-19/1largeRjet1lep/MC/mc_361106.Zee.1largeRjet1lep.root using atlasopenmagic

First import the atlasopenmagic module

In [ ]:
import atlasopenmagic as atom
import uproot

Looking at the filename, the date 2020-08-19 in the filename tells you about the release name and the mc_361106 tells you about the dataset name.

First lets use the atlasopenmagic (atom) attribute available_releases to find the release numbers

In [ ]:
atom.available_releases()

We see that the 2020 release is 2020e-13tev. Lets's set the release to this!

In [ ]:
atom.set_release("2020e-13tev")

Now lets see what datasets are available for this release by using the available_datasets() attribute

In [ ]:
atom.available_datasets()

We see that 361106 is there as we would expect! We can now use now use the attribute get_urls. We will intentionally pass less than the required number of arguments for this attribute, because the error will provide useful information such as the skim names within the dataset. A skim is essentially the name of a particular type of particle signature that has been recorded by the detector. Lets looks at the skims for dataset 361106.

In [ ]:
atom.get_urls(361106)

We get an error as expected! Also as expected we see that the skim 1largeRjet1lep is available. We now have all the information to find the file we are looking for. Use the attribute get_urls, but now provide the skim name and the data transfer protocol we want to use.

In [ ]:
atom.get_urls(361106, skim="1largeRjet1lep", protocol="https")

We have our file! The attribute get_urls (when used correctly) returns a list of associated files. There could be multiple files associated with a dataset and skim. In this case there is just one! So to access the root file we have to specify the first element of the list ([0])

So now we can access the root file with uproot like we did before:

In [ ]:
events_using_http_string = uproot.open("http://opendata.cern.ch/eos/opendata/atlas/OutreachDatasets/2020-08-19/1largeRjet1lep/MC/mc_361106.Zee.1largeRjet1lep.root")

is equivalent to the following using atlasopenmagic:

In [ ]:
events_using_atom = uproot.open(atom.get_urls(361106, skim="1largeRjet1lep", protocol="https")[0])

However we are still using the protocol https, so we will still experience errors running over large files! Well opendata uses the root:// protocol by default. If you don't specify a protcol when using get_urls you will use root://


#### Solution 1: root://

In [ ]:
atom.get_urls(361106, skim="1largeRjet1lep")

So always use get_urls without specifying https eg.

In [ ]:
events_using_atom = uproot.open(atom.get_urls(361106, skim="1largeRjet1lep")[0])

Using this method you should be able to access data at cern without any errors!

Now lets access the internals of the root file as usual. Use the attribute keys to find the name of the TTree

In [ ]:
events_using_atom.keys()

Access the TTree using its name ['mini'] and then see what it contains using the attribute keys() 

In [ ]:
events_using_atom['mini'].keys()

You can now access the specific branches of the TTree that you want

In [ ]:
sel_atom_events = events_using_atom['mini'].arrays(["lep_n", "lep_charge", "lep_type", "lep_pt", "lep_eta", "lep_phi", "lep_E"])

#### Solution 2. Local disk access

You can download the files over http with no issues, so long as the http transfer is not via uproot. So on linux you could use programs like wget or curl to download the files via http. For instance, lets look at the file https://opendata.cern.ch/eos/opendata/atlas/OutreachDatasets/2020-08-19/1largeRjet1lep/MC/mc_361106.Zee.1largeRjet1lep.root again. This file can be downloaded to the local disk and accessed directly. So for the above root file, it must be put in a directory tree that follows the path OutreachDatasets/2020-08-19/1largeRjet1lep/MC. In this directory you would put the file mc_361106.Zee.1largeRjet1lep.root. 

Warning! This example is specific to the Monty servers at RAL and will most likely fail everywhere else!

On Monty we put all the files under the directory /home/jupyter-datastore/data/opendata/atlas. So the local file can we access by doing the following

In [ ]:
events_local_http_string = uproot.open("/home/jupyter-datastore/data/opendata/atlas/OutreachDatasets/2020-08-19/1largeRjet1lep/MC/mc_361106.Zee.1largeRjet1lep.root")

You have to be careful that the local file exists. If the file doesn't exist and is required locally, you may have to put in a request for the file to be downloaded locally

### Further Reading

https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/13-TeV-examples/uproot_python/MetadataTutorial.ipynb

In [ ]:
!pip show atlasopenmagic

In [ ]:
!atom -h